# 03: Build the Model Dataset (2025 SHED)

This notebook runs **once** and writes a single model-ready dataset that every modelling notebook loads, so that the logistic regression, tree-based and gradient-boosted models all use the same rows, target, features and train/test split. Model comparisons are only fair when those are identical.

**Input:** `data/processed/public2025_clean.csv` (from `01_clean_survey.ipynb`)

**Outputs:**

| File | What's in it |
|---|---|
| `data/processed/model_dataset_2025.csv` | One row per respondent: `shedid`, the target, `weight_pop`, `split` (train/test), `cv_fold`, and every feature column (cleaned and **not** encoded) |
| `data/processed/model_dataset_2025_spec.json` | The column lists: which columns are features, which are numeric / categorical / ordinal (with their order), protected attributes, the EF-module columns, and what was excluded and why |

Features are saved **unencoded** on purpose. Each model family wants different preprocessing (logistic regression needs one-hot + scaling, trees don't need scaling, K-prototypes wants raw categories), and encoders must be fitted on the training rows only, so encoding belongs inside each model's `Pipeline`.

Re-run this notebook only when the cleaning or the feature definitions change, and tell the team when you do. The loading snippet is at the bottom.

## 1. Imports and configuration

In [1]:
import json
from datetime import date

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

In [2]:
# Create a constant train-test split
RANDOM_STATE = 42
TEST_SIZE = 0.20

# Create a constant number for cross validation folds.
# I think if we made individual folds for different models, 
# we might not be able to fairly assess whether that model is better or 
# worse as they might get lucky and a model could look better because they got an 
# easier split

# I settled for 5 but I like the idea of 10 folds too. 
# However, that would require a lot of compute.
N_CV_FOLDS = 5

ID_COLUMN = "shedid"
TARGET_SOURCE = "pay_casheqv"   # "Could cover a $400 expense with cash or its equivalent" (Yes / No)
TARGET = "financially_fragile"  # 1 = could NOT cover it, 0 = could
WEIGHT_COLUMN = "weight_pop"    # population weight (reproduces the Fed's published figures)

## 2. Load the cleaned survey

Run **one** of the two cells below: 2.1 (a) when running locally, 2.1 (b) in Colab.

In [3]:
# 2.1 (a) Load data file locally
DATA_PATH = "../data/processed/public2025_clean.csv"

In [ ]:
# 2.1 (b) Load data file on Colab
from google.colab import files

files.upload()

DATA_PATH = "/content/public2025_clean.csv"

In [5]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {DATA_PATH} \nRows: {df.shape[0]:,}\nColumns: {df.shape[1]:,}")

Loaded ../data/processed/public2025_clean.csv 
Rows: 12,934
Columns: 809


In [6]:
# Run data validations quickly

assert len(df) == 12_934, "Expected all 12,934 respondents"
assert df[ID_COLUMN].is_unique, "Respondent IDs are duplicated"
assert df[WEIGHT_COLUMN].notna().all() and (df[WEIGHT_COLUMN] > 0).all(), "Population weights must be present and positive"
assert set(df[TARGET_SOURCE].unique()) == {"Yes", "No"}, "Unexpected values in the target source"
print("Data validation passed.")

Data validation passed.


## 3. Target

`financially_fragile = 1` when the respondent answered **No** to `pay_casheqv`. The replication check from `02_replicate_fed_estimate.ipynb` is repeated: if the weighted share that *can* cover $400 is more than 0.5 points from the Fed's published 63%, the notebook stops.

In [7]:
y = (df[TARGET_SOURCE] == "No").astype(int).rename(TARGET)

weighted_can_cover = 1 - np.average(y, weights=df[WEIGHT_COLUMN])
print(f"Can cover $400, weighted:   {weighted_can_cover:.2%}   (Fed published: 63%)")
print(f"Fragile, unweighted:        {y.mean():.2%}")

assert abs(weighted_can_cover * 100 - 63.0) <= 0.5, "Replication check failed; the data pipeline is off"

Can cover $400, weighted:   63.14%   (Fed published: 63%)
Fragile, unweighted:        35.07%


## 4. Feature columns

Start from every column and remove anything that isn't a legitimate predictor:

| Group | Columns | Why excluded |
|---|---|---|
| IDs and weights | `shedid`, `weight`, `weight_pop`, `panel_weight`, `panel_weight_pop` | Weights are used to *fit and evaluate*, never as predictors (`shedid` and `weight_pop` are still saved, just not as features) |
| Target | `pay_casheqv` | It's the label |
| Target derivation | `EF3_a` to `EF3_h` | How the respondent would pay the $400. `pay_casheqv` is built from these, so they are direct leakage and are always excluded. |
| Imputation flags | `*_iflag` | Describe imputation, not the household (`K0_iflag` predicting `K0` is leakage) |
| Survey metadata | `pp*date` profile-survey dates, `dov_ef7`, `DOV_X11` | Survey administration rather than household characteristics (`dov_*` look like display-order flags; confirm in the codebook) |
| State geography | `ppstaten` | The Fed says the survey isn't designed for state-level estimates; `ppreg4` / `ppreg9` stay in |

The rest of the emergency-savings module (`EF1`, `EF2`, `EF5C`, `EF5D`, `EF7`, `EF7A`, `EF8_*`) **is** kept, but listed separately in the spec as `ef_module`. Those questions aren't derived from the target, but they are very close to it (`EF2` alone classifies 82% correctly vs. a 65% majority rate), so each model notebook can decide whether to drop them.

In [8]:
iflag_columns = [c for c in df.columns if c.endswith("_iflag")]

ef3_columns = [
    col for col in df.columns
    if col.startswith("EF3_") and not col.endswith("_iflag")
]

survey_date_columns = [
    col for col in df.columns
    if col.startswith("pp") and col.endswith("date")
]

In [9]:
EXCLUDED = {
    "id_and_weights": [
        "shedid",
        "weight",
        "weight_pop",
        "panel_weight",
        "panel_weight_pop",
    ],

    "target": [
        TARGET_SOURCE,
    ],

    "ef3_cols": ef3_columns,

    "imputation_flags": iflag_columns,

    "survey_metadata": [
        "duration",
        "dov_ef7",
        "DOV_X11",
        *survey_date_columns,
    ],

    "state_geography": [
        "ppstaten",
    ],
}

# Only keep excluded columns that actually exist in the dataset
for group in EXCLUDED:
    EXCLUDED[group] = [
        col for col in EXCLUDED[group]
        if col in df.columns
    ]

print("Excluded columns")
print("----------------")
for group, columns in EXCLUDED.items():
    print(f"{group:<18} {len(columns):>4}")

Excluded columns
----------------
id_and_weights        5
target                1
ef3_cols              8
imputation_flags    357
survey_metadata      12
state_geography       1


In [10]:
# Build the feature list
excluded_columns = set()

for columns in EXCLUDED.values():
    excluded_columns.update(columns)

FEATURES = [
    col for col in df.columns
    if col not in excluded_columns
]

print(f"Features: {len(FEATURES)}")

Features: 425


In [11]:
# Emergency-finance module features
EF_MODULE = [
    col for col in FEATURES
    if col.startswith("EF")
]


PROTECTED_ATTRIBUTES = [
    "ppracem",       # race
    "ppethm",        # ethnicity
    "race_5cat",     # race category
    "pphispan",      # Hispanic ethnicity

    "ppgender",      # gender
    "malefemale",    # gender

    "ppagecat",      # age group

    "pppa_lgb",      # sexual orientation
]

print(f"EF module features: {len(EF_MODULE)}")

EF module features: 16


In [12]:
# Leakage tests
assert TARGET_SOURCE not in FEATURES
assert not any(c.startswith("EF3_") for c in FEATURES), "EF3 target-derivation columns leaked in"
assert not any(c.endswith("_iflag") for c in FEATURES), "Imputation flags leaked in"
assert not set(FEATURES) & set(EXCLUDED["id_and_weights"]), "IDs or weights leaked in"
assert set(PROTECTED_ATTRIBUTES).issubset(FEATURES), "Protected attributes should stay in the features for auditing"
print("Leakage checks passed.")

Leakage checks passed.


## 5. Clean values and assign feature types

- **Ordinal (2):** `B2` (financial well-being) and `inc_4cat_50k` (income bracket), with their natural order, as in `01_clean_survey.ipynb`
- **Numeric:** columns stored as numbers. Several (`CG2`, `R3`, `M4`, `BK49A`, `BK49B`, `ppsi2302`) are only asked of some respondents, so they are mostly blank; the blanks are left as `NaN` for each model's imputer
- **Categorical:** everything else. `__NOT_ASKED__` (skip logic) stays as its own category

Text values are stripped of leading/trailing spaces (some columns have `" Yes"` alongside `"Yes"`) and empty strings become `NaN`.

In [13]:
# Separating the features by type

X = df[FEATURES].copy()

NUMERIC_FEATURES = (
    X.select_dtypes(include=np.number)
    .columns
    .tolist()
)

ORDINAL_FEATURES = ["B2", "inc_4cat_50k"]

# Natural order of each ordinal feature, lowest to highest
ORDINAL_ORDER = {
    "B2": [
        "Finding it difficult to get by",
        "Just getting by",
        "Doing okay",
        "Living comfortably",
    ],
    "inc_4cat_50k": [
        "Less than $25,000",
        "$25,000–$49,999",
        "$50,000–$99,999",
        "$100,000 or more",
    ],
}

CATEGORICAL_FEATURES = [
    column
    for column in FEATURES
    if column not in NUMERIC_FEATURES
    and column not in ORDINAL_FEATURES
]

print("Feature types")
print("-------------")
print(f"Numeric:     {len(NUMERIC_FEATURES)}")
print(f"Categorical: {len(CATEGORICAL_FEATURES)}")
print(f"Ordinal:     {len(ORDINAL_FEATURES)}")

Feature types
-------------
Numeric:     11
Categorical: 412
Ordinal:     2


In [14]:
# Clean categorical texts

TEXT_FEATURES = CATEGORICAL_FEATURES + ORDINAL_FEATURES

for column in TEXT_FEATURES:
    X[column] = (
        X[column]
        .astype("object")
        .str.strip()
        .replace("", np.nan)
    )

# Every ordinal value must appear in its order list
for column, order in ORDINAL_ORDER.items():
    unexpected = set(X[column].dropna()) - set(order)
    assert not unexpected, f"{column} has values outside its order: {unexpected}"

In [15]:
classified_feature_count = (
    len(NUMERIC_FEATURES)
    + len(CATEGORICAL_FEATURES)
    + len(ORDINAL_FEATURES)
)

assert classified_feature_count == len(FEATURES)
print(f"{classified_feature_count} = {len(FEATURES)}")

425 = 425


## 6. Train/test split and cross-validation folds

The split is saved as a column so every model uses exactly the same held-out respondents.

- `split`: `"train"` / `"test"`, 80/20 stratified on the target with `random_state=42`. These are the same settings as `02_replicate_fed_estimate.ipynb`
- `cv_fold`: `0` to `4` for training rows (stratified 5-fold, for tuning and for calibration in October), `-1` for test rows. Using shared folds keeps cross-validation scores comparable across models

The test set is for final evaluation only. Tune on `cv_fold`, not on `test`.

In [16]:
train_index, test_index = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Label every row as train or test. The name becomes the CSV column name.
split = pd.Series("train", index=df.index, name="split")
split.loc[test_index] = "test"

print(f"Train rows: {len(train_index):,}")
print(f"Test rows:  {len(test_index):,}")

Train rows: 10,347
Test rows:  2,587


In [17]:
cv_fold = pd.Series(
    -1,
    index=df.index,
    name="cv_fold",
    dtype="int64",
)

folds = StratifiedKFold(
    n_splits=N_CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

for fold_number, (_, validation_positions) in enumerate(
    folds.split(train_index, y.loc[train_index])
):
    validation_index = train_index[validation_positions]
    cv_fold.loc[validation_index] = fold_number

In [18]:
assert (cv_fold[test_index] == -1).all()
assert (cv_fold[train_index] >= 0).all()
print("Cross-validation folds assigned correctly.")

Cross-validation folds assigned correctly.


In [19]:
# Get Summaries

summary = pd.DataFrame({
    "split": split,
    "cv_fold": cv_fold,
    TARGET: y,
})

summary_table = (
    summary
    .groupby(["split", "cv_fold"])
    .agg(
        rows=(TARGET, "size"),
        fragile_rate=(TARGET, "mean"),
    )
    .round(4)
)

print(summary_table)

               rows  fragile_rate
split cv_fold                    
test  -1       2587        0.3506
train  0       2070        0.3507
       1       2070        0.3507
       2       2069        0.3509
       3       2069        0.3509
       4       2069        0.3504


## 7. Assemble and save

In [20]:
dataset = pd.concat(
    [df[[ID_COLUMN]], y, df[[WEIGHT_COLUMN]], split, cv_fold, X],
    axis=1,
)

In [21]:
# Describe the dataset so model notebooks don't hard-code column lists
spec = {
    "dataset": "model_dataset_2025",
    "created": date.today().isoformat(),

    "id": ID_COLUMN,

    "target": {
        "column": TARGET,
        "source": TARGET_SOURCE,
        "definition": (
            f"1 if {TARGET_SOURCE} == 'No', else 0"
        ),
    },

    "weight": WEIGHT_COLUMN,

    "split": {
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "stratified_on": TARGET,
    },

    "cv": {
        "n_folds": N_CV_FOLDS,
        "random_state": RANDOM_STATE,
        "test_fold": -1,
    },

    "features": {
        "numeric": NUMERIC_FEATURES,
        "categorical": CATEGORICAL_FEATURES,
        "ordinal": ORDINAL_FEATURES,
        "ordinal_order": ORDINAL_ORDER,
    },

    "ef_module": EF_MODULE,
    "protected_attributes": PROTECTED_ATTRIBUTES,

    "excluded": EXCLUDED,
}

In [22]:
# Save the dataset and its spec
dataset_path = "../data/processed/model_dataset_2025.csv"
spec_path = "../data/processed/model_dataset_2025_spec.json"

dataset.to_csv(dataset_path, index=False)
with open(spec_path, "w", encoding="utf-8") as f:
    json.dump(spec, f, indent=2, ensure_ascii=False)

print(f"Wrote {dataset_path}  ({dataset.shape[0]:,} rows x {dataset.shape[1]:,} columns)")
print(f"Wrote {spec_path}")

Wrote ../data/processed/model_dataset_2025.csv  (12,934 rows x 430 columns)
Wrote ../data/processed/model_dataset_2025_spec.json


## 8. Read-back check

Reload the CSV and confirm the key columns survived with the right names and values.

In [23]:
reloaded = pd.read_csv(dataset_path, low_memory=False)

assert reloaded.shape == dataset.shape, "Row or column count changed"
assert {ID_COLUMN, TARGET, WEIGHT_COLUMN, "split", "cv_fold"}.issubset(reloaded.columns), \
    "A key column is missing or misnamed"
assert reloaded["split"].isin(["train", "test"]).all(), "Unexpected values in split"
assert (reloaded.loc[reloaded["split"] == "test", "cv_fold"] == -1).all(), "Test rows must have cv_fold = -1"

print("Read-back check passed.")

Read-back check passed.


## 9. How to use it in a model notebook

Copy this into the top of any modelling notebook (in Colab, upload both files from `data/processed/` first):

```python
import json
import pandas as pd

data = pd.read_csv("../data/processed/model_dataset_2025.csv", low_memory=False)
with open("../data/processed/model_dataset_2025_spec.json", encoding="utf-8") as f:
    spec = json.load(f)

TARGET = spec["target"]["column"]
WEIGHT = spec["weight"]
FEATURES = (
    spec["features"]["numeric"]
    + spec["features"]["categorical"]
    + spec["features"]["ordinal"]
)

train = data[data["split"] == "train"]
test = data[data["split"] == "test"]

X_train, y_train, w_train = train[FEATURES], train[TARGET], train[WEIGHT]
X_test, y_test, w_test = test[FEATURES], test[TARGET], test[WEIGHT]
```

Also in the spec:
- `spec["ef_module"]`: the emergency-savings questions. To drop them: `FEATURES = [c for c in FEATURES if c not in spec["ef_module"]]`
- `spec["protected_attributes"]`: columns for the subgroup (fairness) audit
- `spec["features"]["ordinal_order"]`: value order for each ordinal feature, e.g. for `OrdinalEncoder(categories=...)`